In [0]:
%python
df_taxa_bruta_nat_valid = spark.read.table("workspace.silver.taxa_bruta_natalidade")
df_esp_vida_nasc_valid = spark.read.table("workspace.silver.esperanca_vida_nascer")

In [0]:
display(df_taxa_bruta_nat_valid.limit(10))

### Aplicando regras para a camada de Validação

Regras:
- 1 - Deve ser uma das 27 siglas válidas
- 2 - Deve estar entre 1 e 12 (meses)
- 3 - Deve estar entre os 27 códigos (códigos definidos pelo IBGE para cada estado)
- 4 - Deve ser uma das 5 regiões (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)
- 5 - Deve estar entre 1 e 5 (5 regiões do Brasil)
- 6 - Deve ser uma das siglas válidas (siglas das regiões)
- 7 - Não pode ser nula porque é uma Data de referência do indicador, então considera obrigatória
- 8 - Indicador UF não deve ser nulo e deve ser maior que zero
- 9 - Indicador REG não deve ser nulo e deve ser maior que zero
- 10 - Indicador BR não deve ser nulo e deve ser maior que zero
- 11 - Validar correspondência co_uf ↔ sg_uf
- 12 - Validar coerência hierárquica regional (co_regiao_brasil deve corresponder à região da UF)
- 13 - Validar consistência dt_competencia ↔ co_ano/co_mes
- 14 - Validar range de anos razoáveis (1900 ≤ ano ≤ 2100)
- 15 - Nome da UF válido (tolera variações de caixa/acento/espaços, rejeita Inconsistência)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

uf_valida = {
    "AC": 12, "AL": 27, "AM": 13, "AP": 16, "BA": 29, "CE": 23, "DF": 53,
    "ES": 32, "GO": 52, "MA": 21, "MG": 31, "MS": 50, "MT": 51, "PA": 15,
    "PB": 25, "PE": 26, "PI": 22, "PR": 41, "RJ": 33, "RN": 24, "RO": 11,
    "RR": 14, "RS": 43, "SC": 42, "SE": 28, "SP": 35, "TO": 17
}
codigo_uf_valida = list(uf_valida.values())
regiao_valida = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
sigla_regiao_valida = ["N", "NE", "SE", "S", "CO"]
mapa_regiao = {
    1: ("Norte", "N"),
    2: ("Nordeste", "NE"),
    3: ("Sudeste", "SE"),
    4: ("Sul", "S"),
    5: ("Centro-Oeste", "CO")
}
uf_regiao = {
    12: 1, 27: 2, 13: 1, 16: 1, 29: 2, 23: 2, 53: 5, 32: 3, 52: 5, 21: 2, 31: 3, 50: 5, 51: 5, 15: 1,
    25: 2, 26: 2, 22: 2, 41: 4, 33: 3, 24: 2, 11: 1, 14: 1, 43: 4, 42: 4, 28: 2, 35: 3, 17: 1
}

# Regra 15
uf_nomes = {
    11: "RONDONIA", 12: "ACRE", 13: "AMAZONAS", 14: "RORAIMA", 15: "PARA",
    16: "AMAPA", 17: "TOCANTINS", 21: "MARANHAO", 22: "PIAUI", 23: "CEARA",
    24: "RIOGRANDEDONORTE", 25: "PARAIBA", 26: "PERNAMBUCO", 27: "ALAGOAS",
    28: "SERGIPE", 29: "BAHIA", 31: "MINASGERAIS", 32: "ESPIRITOSANTO",
    33: "RIODEJANEIRO", 35: "SAOPAULO", 41: "PARANA", 42: "SANTACATARINA",
    43: "RIOGRANDEDOSUL", 50: "MATOGROSSODOSUL", 51: "MATOGROSSO",
    52: "GOIAS", 53: "DISTRITOFEDERAL"
}

regras = [
    # Regra 1
    ((~F.upper(F.col("sg_uf")).isin(*list(uf_valida.keys()))) | (F.col("sg_uf").isNull()), "UF inválida"),
    
    # Regra 2
    ((~F.col("co_mes").between(1, 12)) | (F.col("co_mes").isNull()), "Mês inválido"),
    
    # Regra 3
    ((~F.col("co_uf").isin(*codigo_uf_valida)) | (F.col("co_uf").isNull()), "Código UF inválido"),
    
    # Regra 4
    ((~F.col("no_regiao_brasil").isin(*regiao_valida)) | (F.col("no_regiao_brasil").isNull()), "Região inválida"),

    # Regra 5
    ((~F.col("co_regiao_brasil").between(1, 5)) | (F.col("co_regiao_brasil").isNull()), "Código região inválido"),
    
    # Regra 6
    ((~F.upper(F.col("sg_regiao_brasil")).isin(*sigla_regiao_valida)) | (F.col("sg_regiao_brasil").isNull()), "Sigla região inválida"),
    
    # Regra 7
    (F.col("dt_competencia").isNull(), "Data competência inválida"),
    
    # Regra 8
    ((F.col("vl_indicador_calculado_uf").isNull()) | (F.col("vl_indicador_calculado_uf") <= 0), "Indicador UF calculado inválido"),
    
    # Regra 9
    ((F.col("vl_indicador_calculado_reg").isNull()) | (F.col("vl_indicador_calculado_reg") <= 0), "Indicador REG calculado inválido"),
    
    # Regra 10
    ((F.col("vl_indicador_calculado_br").isNull()) | (F.col("vl_indicador_calculado_br") <= 0), "Indicador BR calculado inválido"),
    
    # Regra 11
    (F.col("co_uf") != F.expr(f"CASE sg_uf {' '.join([f'WHEN \"{k}\" THEN {v}' for k, v in uf_valida.items()])} END"), "Correspondência co_uf e sg_uf inválida"),
    
    # Regra 12
    (F.col("co_regiao_brasil") != F.expr(f"CASE co_uf {' '.join([f'WHEN {k} THEN {v}' for k, v in uf_regiao.items()])} END"), "Coerência regional UF → Região inválida"),
    
    # Regra 13
    ((F.year(F.to_date(F.col("dt_competencia"))) != F.col("co_ano")) | (F.month(F.to_date(F.col("dt_competencia"))) != F.col("co_mes")), "Consistência dt_competencia e co_ano/co_mes inválida"),
    
    # Regra 14
    ((F.col("co_ano") < 1900) | (F.col("co_ano") > 2100), "Ano fora do range razoável"),
    
    # Regra 15
    (  
        (F.col("no_uf").isNotNull()) & 
        (~F.expr(f"""
            array_contains(
                split(CASE co_uf {' '.join([f'WHEN {co} THEN \"{nome}|{[k for k,v in uf_valida.items() if v==co][0]}\"' for co, nome in uf_nomes.items()])} END, '[|]'),
                regexp_replace(upper(translate(no_uf, 'áàâãäéêëíóôõöúüçÁÀÂÃÄÉÊËÍÓÔÕÖÚÜÇ', 'aaaaaeeeiooooouucAAAAAEEEIOOOOOUUC')), '[ .-]', '')
            )
        """)),
        "Nome da UF inválido ou corrompido"
    )
]

# Função para aplicar as regras e gerar os motivos de rejeição
def aplicar_motivos_rejeicao(df, regras):
    return df.withColumn(
        "motivos_rejeicao",
        F.filter(
            F.array(
                *[
                    F.when(condicao, mensagem)
                    for condicao, mensagem in regras
                ]
            ),
            lambda x: x.isNotNull()
        )
    )

df_taxa_bruta_nat_valid = aplicar_motivos_rejeicao(df_taxa_bruta_nat_valid, regras)
df_esp_vida_nasc_valid = aplicar_motivos_rejeicao(df_esp_vida_nasc_valid, regras)

In [0]:
dados_teste = [

    # Registro totalmente válido
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # UF inválida (Regra 1)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "XX",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # Mês inválido (Regra 2)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 13,
        "col_ds_origem": "TESTE"
    },

    # Código UF inválido (Regra 3)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 99,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # Região inválida (Regra 4)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Leste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # Correspondência co_uf ↔ sg_uf inválida (Regra 11)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 25,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.91,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # Coerência regional UF → Região inválida (Regra 12)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 25,
        "no_municipio": None,
        "sg_uf": "PB",
        "no_uf": "Paraíba",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Nordeste",
        "sg_regiao_brasil": "NE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 16.88,
        "vl_indicador_calculado_reg": 17.99,
        "vl_indicador_calculado_br": 15.96,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2007-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2007,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # Indicador UF negativo (Regra 8)
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 14,
        "no_municipio": None,
        "sg_uf": "RR",
        "no_uf": "Roraima",
        "co_regiao_brasil": 1,
        "no_regiao_brasil": "Norte",
        "sg_regiao_brasil": "N",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": -27.78,
        "vl_indicador_calculado_reg": 25.70,
        "vl_indicador_calculado_br": 18.31,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2002-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2002,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 1 - UF válida da região Sul
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 42,
        "no_municipio": None,
        "sg_uf": "SC",
        "no_uf": "Santa Catarina",
        "co_regiao_brasil": 4,
        "no_regiao_brasil": "Sul",
        "sg_regiao_brasil": "S",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 11.25,
        "vl_indicador_calculado_reg": 11.80,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-01-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 1,
        "col_ds_origem": "TESTE"
    },

    # 2 - UF válida da região Norte
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 13,
        "no_municipio": None,
        "sg_uf": "AM",
        "no_uf": "Amazonas",
        "co_regiao_brasil": 1,
        "no_regiao_brasil": "Norte",
        "sg_regiao_brasil": "N",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 18.10,
        "vl_indicador_calculado_reg": 17.50,
        "vl_indicador_calculado_br": 15.20,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2018-06-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2018,
        "co_mes": 6,
        "col_ds_origem": "TESTE"
    },

    # 3 - Indicador UF zerado
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 0.0,
        "vl_indicador_calculado_reg": 12.68,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2021-03-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2021,
        "co_mes": 3,
        "col_ds_origem": "TESTE"
    },

    # 4 - Indicador regional negativo
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": -2.0,
        "vl_indicador_calculado_br": 13.81,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 5 - Indicador Brasil negativo
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": -1.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 6 - Ano incompatível com competência
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 7 - Mês incompatível com competência
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2019,
        "co_mes": 11,
        "col_ds_origem": "TESTE"
    },

    # 8 - Sigla regional inválida
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "XX",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 9 - Nome da UF incompatível
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "Rio de Janeiro",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 10 - Código da região incompatível
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 5,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 11 - Unidade medida inválida
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Percentual",
        "ds_item_categoria": "Total",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 12 - Categoria inválida
    {
        "co_ibge": None,
        "vl_indicador_calculado_mun": None,
        "co_uf": 35,
        "no_municipio": None,
        "sg_uf": "SP",
        "no_uf": "São Paulo",
        "co_regiao_brasil": 3,
        "no_regiao_brasil": "Sudeste",
        "sg_regiao_brasil": "SE",
        "co_regiao_saude": None,
        "no_regiao_saude": None,
        "no_macro": None,
        "co_macro": None,
        "vl_indicador_calculado_rs": None,
        "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 12.5,
        "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0,
        "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01",
        "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa",
        "ds_item_categoria": "INVALIDA",
        "co_ano": 2020,
        "co_mes": 12,
        "col_ds_origem": "TESTE"
    },

    # 13 ao 20 - Casos com múltiplas violações
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 99,
        "no_municipio": None, "sg_uf": "XX", "no_uf": "São Paulo",
        "co_regiao_brasil": 3, "no_regiao_brasil": "Sudeste", "sg_regiao_brasil": "SE",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": -10.0, "vl_indicador_calculado_reg": 12.0,
        "vl_indicador_calculado_br": 13.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2020-12-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2020, "co_mes": 13, "col_ds_origem": "TESTE"
    },

    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 25,
        "no_municipio": None, "sg_uf": "SP", "no_uf": "Paraíba",
        "co_regiao_brasil": 5, "no_regiao_brasil": "Nordeste", "sg_regiao_brasil": "N",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": -5.0, "vl_indicador_calculado_reg": -3.0,
        "vl_indicador_calculado_br": -1.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2019-05-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Percentual", "ds_item_categoria": "INVALIDA",
        "co_ano": 2020, "co_mes": 15, "col_ds_origem": "TESTE"
    },

    # 15
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 42,
        "no_municipio": None, "sg_uf": "SC", "no_uf": "Santa Catarina",
        "co_regiao_brasil": 1, "no_regiao_brasil": "Norte", "sg_regiao_brasil": "N",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 9999.99, "vl_indicador_calculado_reg": 9999.99,
        "vl_indicador_calculado_br": 9999.99, "vl_indicador_calculado_al": None,
        "dt_competencia": "2018-08-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2018, "co_mes": 8, "col_ds_origem": "TESTE"
    },

    # 16
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 33,
        "no_municipio": None, "sg_uf": "RJ", "no_uf": "Rio de Janeiro",
        "co_regiao_brasil": 2, "no_regiao_brasil": "Nordeste", "sg_regiao_brasil": "NE",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": -1, "vl_indicador_calculado_reg": -1,
        "vl_indicador_calculado_br": -1, "vl_indicador_calculado_al": None,
        "dt_competencia": "2022-07-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "ERRADA", "ds_item_categoria": "ERRADA",
        "co_ano": 2023, "co_mes": 14, "col_ds_origem": "TESTE"
    },

    # 17
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 11,
        "no_municipio": None, "sg_uf": "RO", "no_uf": "Roraima",
        "co_regiao_brasil": 1, "no_regiao_brasil": "Norte", "sg_regiao_brasil": "N",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 15.0, "vl_indicador_calculado_reg": 14.0,
        "vl_indicador_calculado_br": 13.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2021-02-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2021, "co_mes": 2, "col_ds_origem": "TESTE"
    },

    # 18
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 52,
        "no_municipio": None, "sg_uf": "GO", "no_uf": "Goiás",
        "co_regiao_brasil": 4, "no_regiao_brasil": "Sul", "sg_regiao_brasil": "S",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 10.0, "vl_indicador_calculado_reg": 11.0,
        "vl_indicador_calculado_br": 12.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2017-09-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2017, "co_mes": 9, "col_ds_origem": "TESTE"
    },

    # 19
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 21,
        "no_municipio": None, "sg_uf": "MA", "no_uf": "Maranhão",
        "co_regiao_brasil": 2, "no_regiao_brasil": "Nordeste", "sg_regiao_brasil": "XX",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 20.0, "vl_indicador_calculado_reg": 18.0,
        "vl_indicador_calculado_br": 15.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2016-04-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2016, "co_mes": 4, "col_ds_origem": "TESTE"
    },

    # 20
    {
        "co_ibge": None, "vl_indicador_calculado_mun": None, "co_uf": 50,
        "no_municipio": None, "sg_uf": "MS", "no_uf": "Mato Grosso",
        "co_regiao_brasil": 5, "no_regiao_brasil": "Centro-Oeste", "sg_regiao_brasil": "CO",
        "co_regiao_saude": None, "no_regiao_saude": None, "no_macro": None, "co_macro": None,
        "vl_indicador_calculado_rs": None, "vl_indicador_calculado_ms": None,
        "vl_indicador_calculado_uf": 15.0, "vl_indicador_calculado_reg": 14.0,
        "vl_indicador_calculado_br": 13.0, "vl_indicador_calculado_al": None,
        "dt_competencia": "2023-10-01", "dt_atualizacao": "2026-03-18",
        "ds_unidade_medida": "Taxa", "ds_item_categoria": "Total",
        "co_ano": 2023, "co_mes": 10, "col_ds_origem": "TESTE"
    }

]

# Schema explícito
schema = StructType([
    StructField("co_ibge", IntegerType(), True),
    StructField("vl_indicador_calculado_mun", DoubleType(), True),
    StructField("co_uf", IntegerType(), True),
    StructField("no_municipio", StringType(), True),
    StructField("sg_uf", StringType(), True),
    StructField("no_uf", StringType(), True),
    StructField("co_regiao_brasil", IntegerType(), True),
    StructField("no_regiao_brasil", StringType(), True),
    StructField("sg_regiao_brasil", StringType(), True),
    StructField("co_regiao_saude", IntegerType(), True),
    StructField("no_regiao_saude", StringType(), True),
    StructField("no_macro", StringType(), True),
    StructField("co_macro", IntegerType(), True),
    StructField("vl_indicador_calculado_rs", DoubleType(), True),
    StructField("vl_indicador_calculado_ms", DoubleType(), True),
    StructField("vl_indicador_calculado_uf", DoubleType(), True),
    StructField("vl_indicador_calculado_reg", DoubleType(), True),
    StructField("vl_indicador_calculado_br", DoubleType(), True),
    StructField("vl_indicador_calculado_al", DoubleType(), True),
    StructField("dt_competencia", StringType(), True),
    StructField("dt_atualizacao", StringType(), True),
    StructField("ds_unidade_medida", StringType(), True),
    StructField("ds_item_categoria", StringType(), True),
    StructField("co_ano", IntegerType(), True),
    StructField("co_mes", IntegerType(), True),
    StructField("col_ds_origem", StringType(), True)
])

df_teste = spark.createDataFrame(dados_teste, schema=schema)

# Converte dt_competencia de String para Date
df_teste = df_teste.withColumn("dt_competencia", F.to_date(F.col("dt_competencia")))

# Aplica as regras de validação (adiciona motivos_rejeicao automaticamente)
df_teste = aplicar_motivos_rejeicao(df_teste, regras)

In [0]:
display(df_teste)

In [0]:
df_teste_validos = df_teste.filter(F.size(F.col("motivos_rejeicao")) == 0)
df_teste_rejeitados = df_teste.filter(F.size(F.col("motivos_rejeicao")) > 0)

df_taxa_bruta_validos = df_taxa_bruta_nat_valid.filter(F.size("motivos_rejeicao") == 0)
df_taxa_bruta_rejeitados = df_taxa_bruta_nat_valid.filter(F.size("motivos_rejeicao") > 0)

df_esp_vida_validos = df_esp_vida_nasc_valid.filter(F.size("motivos_rejeicao") == 0)
df_esp_vida_rejeitados = df_esp_vida_nasc_valid.filter(F.size("motivos_rejeicao") > 0)


print(f"Testes: {df_teste_validos.count():,} válidos e {df_teste_rejeitados.count():,} rejeitados")
print(f"Taxa Bruta: {df_taxa_bruta_validos.count():,} válidos e {df_taxa_bruta_rejeitados.count():,} rejeitados")
print(f"Esperança Vida: {df_esp_vida_validos.count():,} válidos e {df_esp_vida_rejeitados.count():,} rejeitados")

In [0]:
display(df_teste_validos)
display(df_teste_rejeitados)

### Salvando camada de validação de regras

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.validation;
USE SCHEMA validation;

In [0]:
df_taxa_bruta_nat_valid.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.validation.taxa_bruta_natalidade")

df_esp_vida_nasc_valid.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.validation.esperanca_vida_nascer")

df_teste.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "True") \
.saveAsTable("workspace.validation.teste_de_validacao")


print("Tabelas (validation) criadas corretamente.")